<a href="https://colab.research.google.com/github/TAUforPython/Graph-MachineLearning/blob/main/utils_ERD2MMD_interactive_graph_visualisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Оптимизированный ERD визуализатор с ИИ-переводом
# Установка зависимостей
!pip install -q networkx matplotlib pandas pillow lxml
!apt-get install -y graphviz graphviz-dev -q
!pip install -q pygraphviz


Reading package lists...
Building dependency tree...
Reading state information...
graphviz is already the newest version (2.42.2-6ubuntu0.1).
The following additional packages will be installed:
  libatk1.0-0 libatk1.0-data libgail-common libgail18 libgtk2.0-0
  libgtk2.0-bin libgtk2.0-common libgvc6-plugins-gtk librsvg2-common
  libxcomposite1 libxdot4
Suggested packages:
  gvfs
The following NEW packages will be installed:
  libatk1.0-0 libatk1.0-data libgail-common libgail18 libgraphviz-dev
  libgtk2.0-0 libgtk2.0-bin libgtk2.0-common libgvc6-plugins-gtk
  librsvg2-common libxcomposite1 libxdot4
0 upgraded, 12 newly installed, 0 to remove and 2 not upgraded.
Need to get 2,496 kB of archives.
After this operation, 7,963 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-0 amd64 2.36.0-3build1 [51.9 kB]
Get:3 http://archive.ubuntu

In [12]:

import re
import xml.etree.ElementTree as ET
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import json
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Any
import dataclasses
from dataclasses import dataclass, field
import time

In [17]:


# ============================================================================
# КОНФИГУРАЦИЯ И КОНСТАНТЫ
# ============================================================================
@dataclass
class Config:
    """Конфигурация визуализатора"""
    max_name_length: int = 30
    short_name_length: int = 25
    max_nodes_display: int = 60
    figsize_large: Tuple[int, int] = (40, 28)
    figsize_medium: Tuple[int, int] = (28, 20)
    figsize_small: Tuple[int, int] = (20, 14)

    # Регулярные выражения
    rx_camel: str = r'[a-z][A-Z]'
    rx_numbers: str = r'\d'
    rx_underscores: str = '_'
    rx_schema: str = r'^.*\.'

    # 🎨 Цветовая схема по категориям таблиц
    colors = {
        'directory': '#FFB3BA',      # Справочники (Spr_, NSI_)
        'log': '#BAFFC9',            # Журналы логов (Log_)
        'patient': '#BAE1FF',        # Пациенты (Kont_, Patient)
        'medical': '#FFFFBA',        # Медицинские записи (Ambul_, Hsp_, Diagnoz)
        'drug': '#E2BAFF',           # Лекарства (Drug_, Medication, RLS_)
        'lab': '#FFDFBA',            # Лаборатория (Lab_, Biomaterial)
        'finance': '#BAFFDF',        # Финансы (Bill_, Payment, Contract)
        'admin': '#FFD1BA',          # Административные (Division_, Room, Staff)
        'report': '#D1BAFF',         # Отчёты (URS_, RS_, Report)
        'service': '#FFBAE1',        # Услуги (Service_, Spr_Uslugi)
        'oms': '#BAFFE1',            # ОМС (OMS_, Oms_)
        'parus': '#E1FFBA',          # Парус интеграция (Parus_)
        'cp': '#FFE1BA',             # Копии справочников (cp_)
        'other': '#E0E0E0'           # Прочие
    }

    # 📐 Доступные layout для графа
    layouts = {
        'circular': 'Circular (круговой)',
        'kamada_kawai': 'Kamada-Kawai (энергетический)',
        'shell': 'Shell (концентрические круги)',
        'spectral': 'Spectral (спектральный)',
        'planar': 'Planar (плоскостной)',
        'random': 'Random (случайный)',
        'bipartite': 'Bipartite (двудольный)'
    }

CONFIG = Config()

# ============================================================================
# 📚 КАТЕГОРИЗАТОР ТАБЛИЦ
# ============================================================================
class TableCategorizer:
    """Категоризатор таблиц по именам"""

    # Паттерны для определения категории
    CATEGORY_PATTERNS = {
        'directory': [
            r'^Spr_', r'^NSI', r'^Sprav_', r'^Dictionary', r'^Handbook',
            r'^Classifier', r'^MKB10', r'^cp_'
        ],
        'log': [
            r'^Log_', r'^LOG_', r'^Audit', r'^History', r'^Changes'
        ],
        'patient': [
            r'^Kont_', r'^Patient', r'^Fizlica', r'^Urlica', r'^Kontragent',
            r'^Person', r'^Family'
        ],
        'medical': [
            r'^Ambul', r'^Hsp', r'^Diagnoz', r'^Hospital', r'^Treatment',
            r'^Sickness', r'^Epicrisis', r'^StatCard'
        ],
        'drug': [
            r'^Drug', r'^Medication', r'^RLS_', r'^Pharmacy', r'^Prescription'
        ],
        'lab': [
            r'^Lab', r'^Biomaterial', r'^Analysis', r'^Research', r'^Sample'
        ],
        'finance': [
            r'^Bill', r'^Payment', r'^Contract', r'^Invoice', r'^Price',
            r'^Tariff', r'^Account', r'^Bank'
        ],
        'admin': [
            r'^Division', r'^Room', r'^Staff', r'^Doctor', r'^Department',
            r'^Building', r'^Bed', r'^Schedule', r'^Access'
        ],
        'report': [
            r'^URS_', r'^RS_', r'^Report', r'^Template', r'^FastReport'
        ],
        'service': [
            r'^Service', r'^Spr_Uslugi', r'^Okaz_Uslugi', r'^OMSService'
        ],
        'oms': [
            r'^OMS', r'^Oms', r'^OMSMES', r'^OMSPolicy'
        ],
        'parus': [
            r'^Parus_'
        ],
    }

    @staticmethod
    def categorize(table_name: str) -> str:
        """Определение категории таблицы по имени"""
        name_upper = table_name.upper()

        for category, patterns in TableCategorizer.CATEGORY_PATTERNS.items():
            for pattern in patterns:
                if re.search(pattern, name_upper, re.IGNORECASE):
                    return category

        return 'other'

    @staticmethod
    def get_category_stats(entities: Dict[str, Any]) -> Dict[str, int]:
        """Подсчёт количества таблиц по категориям"""
        stats = defaultdict(int)
        for entity in entities.values():
            category = TableCategorizer.categorize(entity.name)
            stats[category] += 1
        return dict(stats)

# ============================================================================
# 🌐 ИНТЕЛЛЕКТУАЛЬНЫЙ ПЕРЕВОДЧИК (ИСПРАВЛЕННЫЙ)
# ============================================================================
class BilingualTranslator:
    """Переводчик для двуязычного отображения названий"""
    __slots__ = ('rus_to_eng', 'eng_to_rus', 'eng_abbr', 'rus_abbr', 'translit_map', 'cache')

    def __init__(self):
        # Прямой словарь (Русский → Английский)
        self.rus_to_eng = {k.lower(): v for k, v in RUSSIAN_TO_ENGLISH.items()}

        # 🔴 ОБРАТНЫЙ словарь (Английский → Русский)
        self.eng_to_rus = {v.lower(): k for k, v in RUSSIAN_TO_ENGLISH.items()}

        # Словари аббревиатур
        self.eng_abbr = ENGLISH_ABBREVIATIONS
        self.rus_abbr = {k.lower(): v for k, v in RUSSIAN_ABBREVIATIONS.items()}

        # 🔴 Словарь транслитерации (транслит → кириллица)
        self.translit_map = {
            # Справочники
            'spr': 'справочник', 'sprav': 'справочник', 'dir': 'справочник',
            # Посещения
            'poseshenie': 'посещение', 'posesh': 'посещение', 'visit': 'посещение',
            'new': 'новое', 'plan': 'плановое',
            # Услуги
            'uslugi': 'услуги', 'usluga': 'услуга', 'service': 'услуга',
            # Госпитализация
            'hsp': 'госпитализация', 'gosp': 'госпитализация', 'hospital': 'госпиталь',
            'request': 'запрос',
            # Карты
            'kart': 'карта', 'card': 'карта', 'ambul': 'амбулаторная',
            # Диагнозы
            'diagnoz': 'диагноз', 'diagnozy': 'диагнозы', 'diagnosis': 'диагноз',
            'mkb': 'мкб', 'icd': 'мкб', 'znach': 'значение', 'value': 'значение',
            # Лекарства
            'drug': 'лекарство', 'drugs': 'лекарства', 'preparat': 'препарат',
            'medication': 'препарат', 'recept': 'рецепт', 'prescription': 'рецепт',
            'apteka': 'аптека', 'pharmacy': 'аптека', 'sklad': 'склад',
            # Лаборатория
            'lab': 'лаборатория', 'laboratory': 'лаборатория', 'analiz': 'анализ',
            'analysis': 'анализ', 'rezultat': 'результат', 'result': 'результат',
            'referral': 'направление', 'napravlenie': 'направление',
            # Пациенты
            'patient': 'пациент', 'pacient': 'пациент', 'fizlitso': 'физлицо',
            'person': 'физлицо', 'yurlitso': 'юрлицо', 'kontragent': 'контрагент',
            # Сотрудники
            'doctor': 'доктор', 'vrach': 'врач', 'physician': 'врач',
            'sotrudnik': 'сотрудник', 'employee': 'сотрудник', 'staff': 'персонал',
            'user': 'пользователь', 'polzovatel': 'пользователь',
            # Административные
            'otdelenie': 'отделение', 'department': 'отделение',
            'podrazdelenie': 'подразделение', 'division': 'подразделение',
            'palata': 'палата', 'ward': 'палата', 'koyka': 'койка', 'bed': 'койка',
            'room': 'помещение', 'building': 'здание',
            # Финансы
            'schet': 'счет', 'bill': 'счет', 'oplata': 'оплата', 'payment': 'оплата',
            'dogovor': 'договор', 'contract': 'договор', 'tsena': 'цена',
            'price': 'цена', 'tarif': 'тариф', 'tariff': 'тариф',
            'kassa': 'касса', 'bank': 'банк',
            # Документы
            'dokument': 'документ', 'document': 'документ', 'zhurnal': 'журнал',
            'journal': 'журнал', 'otchet': 'отчет', 'report': 'отчет',
            'log': 'журнал', 'ch': 'изменения', 'changes': 'изменения',
            # ОМС
            'oms': 'омс', 'mes': 'мээк', 'policy': 'полис',
            # Общие
            'status': 'статус', 'type': 'тип', 'kind': 'вид', 'category': 'категория',
            'gruppa': 'группа', 'group': 'группа', 'classifier': 'классификатор',
            'reason': 'причина', 'prichina': 'причина', 'template': 'шаблон',
            'shablon': 'шаблон', 'schedule': 'расписание', 'schadule': 'расписание',
        }

        self.cache = {}

    def clean_name(self, name: str) -> str:
        """Очистка имени от схемы"""
        return re.sub(CONFIG.rx_schema, '', name)

    def split_into_parts(self, name: str) -> List[str]:
        """Разделение имени на составные части"""
        if name in self.cache:
            return self.cache[name]

        if '_' in name:
            parts = name.split('_')
        else:
            parts = re.findall(r'[A-Z][a-z]*|[a-z]+', name)

        parts = [p for p in parts if p]
        self.cache[name] = parts
        return parts

    def is_cyrillic(self, text: str) -> bool:
        """Проверка наличия кириллицы"""
        return any(ord(c) > 127 for c in text)

    def is_translit(self, text: str) -> bool:
        """Проверка на транслитерацию"""
        return text.lower() in self.translit_map

    def translate_part(self, part: str) -> Tuple[str, str]:
        """Перевод одной части"""
        part_lower = part.lower()

        if self.is_cyrillic(part):
            russian = part
            english = self.rus_to_eng.get(part_lower, part.capitalize())
            return english, russian

        elif self.is_translit(part):
            russian = self.translit_map.get(part_lower, part)
            english = self.rus_to_eng.get(russian, part.capitalize())
            return english, russian

        else:
            english = part.capitalize()
            russian = self.eng_to_rus.get(part_lower,
                         self.translit_map.get(part_lower, part))
            return english, russian

    def translate_name(self, name: str) -> Tuple[str, str]:
        """Полный перевод имени"""
        cleaned = self.clean_name(name)
        parts = self.split_into_parts(cleaned)

        eng_parts = []
        rus_parts = []

        for part in parts:
            eng, rus = self.translate_part(part)
            eng_parts.append(eng)
            rus_parts.append(rus)

        english = ' '.join(eng_parts)
        russian = ' '.join(rus_parts) if rus_parts else cleaned

        return english, russian

    def get_bilingual_name(self, name: str) -> Dict[str, str]:
        """Получение двуязычного имени"""
        english, russian = self.translate_name(name)

        eng_short = english[:CONFIG.short_name_length] + '...' if len(english) > CONFIG.short_name_length else english
        rus_short = russian[:CONFIG.short_name_length] + '...' if len(russian) > CONFIG.short_name_length else russian

        display_name = f"{eng_short} | {rus_short}"

        return {
            'original': name,
            'clean': russian,
            'english': english,
            'russian': russian,
            'english_short': eng_short,
            'russian_short': rus_short,
            'display': display_name
        }

# Словари переводов
RUSSIAN_TO_ENGLISH = {
    'Справочник': 'Directory', 'Пациент': 'Patient', 'Врач': 'Doctor',
    'Посещение': 'Visit', 'Услуга': 'Service', 'Услуги': 'Services',
    'Госпитализация': 'Hospitalization', 'Диагноз': 'Diagnosis',
    'Лекарство': 'Drug', 'Рецепт': 'Prescription', 'Лаборатория': 'Laboratory',
    'Анализ': 'Analysis', 'Результат': 'Result', 'Отделение': 'Department',
    'Подразделение': 'Division', 'Палата': 'Ward', 'Койка': 'Bed',
    'Счет': 'Bill', 'Оплата': 'Payment', 'Договор': 'Contract',
    'Документ': 'Document', 'Журнал': 'Journal', 'Отчет': 'Report',
    'Пользователь': 'User', 'Сотрудник': 'Employee', 'Контрагент': 'Counterparty',
}

ENGLISH_ABBREVIATIONS = {
    'id': 'Id', 'pk': 'Pk', 'fk': 'Fk', 'code': 'Code', 'name': 'Name',
}

RUSSIAN_ABBREVIATIONS = {
    'спр': 'Directory', 'усл': 'Service', 'пос': 'Visit', 'госп': 'Hospitalization',
    'док': 'Document', 'отд': 'Department', 'пал': 'Ward', 'койк': 'Bed',
}

# ============================================================================
# 📊 СУЩНОСТИ И СВЯЗИ
# ============================================================================
@dataclass
class Entity:
    """Сущность базы данных"""
    id: str
    name: str
    bilingual: Dict[str, str] = field(default_factory=dict)
    category: str = 'other'
    x: float = 0
    y: float = 0

    @property
    def display_name(self) -> str:
        return self.bilingual.get('display', self.name)

    @property
    def color(self) -> str:
        return CONFIG.colors.get(self.category, CONFIG.colors['other'])

@dataclass
class Relation:
    """Связь между сущностями"""
    name: str
    type: str
    source_id: str
    target_id: str
    source_name: str = ''
    target_name: str = ''

# ============================================================================
# 🔍 ПАРСЕР ERD
# ============================================================================
class ERDParser:
    """Парсер ERD файлов"""

    def __init__(self, translator: BilingualTranslator):
        self.translator = translator
        self.entities: Dict[str, Entity] = {}
        self.relations: List[Relation] = []

    def parse(self, content: str) -> bool:
        """Парсинг XML содержимого"""
        try:
            root = ET.fromstring(content)

            data_source = root.find('.//data-source')
            if data_source is not None:
                for entity in data_source.findall('entity'):
                    self._parse_entity(entity)

            relations_elem = root.find('relations')
            if relations_elem is not None:
                for relation in relations_elem.findall('relation'):
                    self._parse_relation(relation)

            return True
        except ET.ParseError as e:
            print(f"❌ Ошибка парсинга XML: {e}")
            return False

    def _parse_entity(self, entity_elem):
        """Парсинг одной сущности"""
        entity_id = entity_elem.get('id')
        entity_name = entity_elem.get('name')

        if entity_name:
            bilingual = self.translator.get_bilingual_name(entity_name)
            category = TableCategorizer.categorize(entity_name)
            self.entities[entity_id] = Entity(
                id=entity_id,
                name=entity_name,
                bilingual=bilingual,
                category=category,
                x=float(entity_elem.get('x', 0)),
                y=float(entity_elem.get('y', 0))
            )

    def _parse_relation(self, relation_elem):
        """Парсинг одной связи"""
        pk_ref = relation_elem.get('pk-ref')
        fk_ref = relation_elem.get('fk-ref')

        if pk_ref in self.entities and fk_ref in self.entities:
            source = self.entities[pk_ref]
            target = self.entities[fk_ref]

            self.relations.append(Relation(
                name=relation_elem.get('name', ''),
                type=relation_elem.get('type', ''),
                source_id=pk_ref,
                target_id=fk_ref,
                source_name=source.display_name,
                target_name=target.display_name
            ))

# ============================================================================
# 🔨 ПОСТРОЕНИЕ ГРАФА
# ============================================================================
class GraphBuilder:
    """Построитель графа связей"""

    def __init__(self, parser: ERDParser, max_nodes: int = 60):
        self.parser = parser
        self.max_nodes = max_nodes
        self.graph = nx.DiGraph()
        self.node_sizes = defaultdict(int)
        self._build()

    def _build(self):
        """Построение графа"""
        rel_counter = defaultdict(int)
        rel_info = {}

        for rel in self.parser.relations:
            key = (rel.source_name, rel.target_name)
            rel_counter[key] += 1
            if key not in rel_info:
                rel_info[key] = rel.name

        # Выбор наиболее связанных узлов
        node_degree = defaultdict(int)
        for rel in self.parser.relations:
            node_degree[rel.source_name] += 1
            node_degree[rel.target_name] += 1

        top_nodes = sorted(node_degree.keys(), key=lambda x: node_degree[x],
                          reverse=True)[:self.max_nodes]
        top_nodes_set = set(top_nodes)

        for rel in self.parser.relations:
            source_name = rel.source_name
            target_name = rel.target_name

            if source_name in top_nodes_set or target_name in top_nodes_set:
                self.node_sizes[source_name] += 1
                self.node_sizes[target_name] += 1

                if source_name not in self.graph:
                    entity = self.parser.entities[rel.source_id]
                    self.graph.add_node(source_name, info=entity,
                                       original_id=rel.source_id,
                                       category=entity.category)

                if target_name not in self.graph:
                    entity = self.parser.entities[rel.target_id]
                    self.graph.add_node(target_name, info=entity,
                                       original_id=rel.target_id,
                                       category=entity.category)

        for (source, target), count in rel_counter.items():
            if source in self.graph and target in self.graph:
                self.graph.add_edge(source, target, weight=min(count, 5),
                                   count=count, name=rel_info.get((source, target), ''))

    def get_stats(self) -> Dict:
        """Получение статистики графа"""
        degrees = dict(self.graph.degree())
        top_entities = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:10]

        category_stats = defaultdict(int)
        for node in self.graph.nodes():
            cat = self.graph.nodes[node].get('category', 'other')
            category_stats[cat] += 1

        return {
            'nodes': self.graph.number_of_nodes(),
            'edges': self.graph.number_of_edges(),
            'top_entities': top_entities,
            'category_stats': dict(category_stats)
        }

# ============================================================================
# 🎨 MATPLOTLIB ВИЗУАЛИЗАТОР
# ============================================================================
# ============================================================================
# ВИЗУАЛИЗАТОР MATPLOTLIB (ИСПРАВЛЕННЫЙ)
# ============================================================================
class MatplotlibVisualizer:
    """Визуализация с помощью Matplotlib"""

    @staticmethod
    def create_layout(graph, layout_type: str = 'spring'):
        """Создание layout для графа"""
        if layout_type == 'circular':
            return nx.circular_layout(graph)
        elif layout_type == 'kamada_kawai':
            return nx.kamada_kawai_layout(graph)
        elif layout_type == 'shell':
            return nx.shell_layout(graph)
        elif layout_type == 'spectral':
            return nx.spectral_layout(graph)
        elif layout_type == 'planar':
            return nx.planar_layout(graph)
        elif layout_type == 'random':
            return nx.random_layout(graph)
        elif layout_type == 'bipartite':
            return nx.bipartite_layout(graph, list(graph.nodes())[:len(graph.nodes())//2])
        else:  # spring
            try:
                return nx.nx_agraph.graphviz_layout(graph, prog='dot')
            except:
                return nx.spring_layout(graph, k=3, iterations=50, seed=42)

    @staticmethod
    def visualize(graph, node_sizes, layout_type: str = 'spring',
                 figsize: Tuple[int, int] = CONFIG.figsize_medium):
        """Визуализация графа"""
        fig, ax = plt.subplots(figsize=figsize)

        pos = MatplotlibVisualizer.create_layout(graph, layout_type)

        # Группировка узлов по категориям для раскраски
        node_colors = []
        for node in graph.nodes():
            category = graph.nodes[node].get('category', 'other')
            node_colors.append(CONFIG.colors.get(category, CONFIG.colors['other']))

        sizes = [min(node_sizes.get(node, 1) * 400 + 600, 3500) for node in graph.nodes()]

        # Узлы
        nx.draw_networkx_nodes(graph, pos, node_size=sizes, node_color=node_colors,
                              edgecolors='black', linewidths=1.5, alpha=0.85, ax=ax)

        # Ребра
        edges = graph.edges(data=True)
        widths = [min(d['weight'] * 2, 5) for (_, _, d) in edges]

        nx.draw_networkx_edges(graph, pos, width=widths, alpha=0.4, edge_color='gray',
                              arrows=True, arrowsize=15, connectionstyle='arc3,rad=0.1', ax=ax)

        # Подписи узлов
        for node, (x, y) in pos.items():
            info = graph.nodes[node].get('info')
            if info:
                display_text = info.display_name
                if len(display_text) > 25:
                    parts = display_text.split(' | ')
                    if len(parts) == 2:
                        display_text = f"{parts[0][:15]}... | {parts[1][:15]}..."
            else:
                display_text = node[:25]

            # 🔴 ИСПРАВЛЕНО: убран параметр ax=ax
            ax.text(x, y, display_text, fontsize=6, fontweight='bold',
                   ha='center', va='center',
                   bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                            alpha=0.85, edgecolor='lightgray'))

        ax.set_title(f"ERD Visualization - Layout: {layout_type.upper()}",
                    fontsize=14, fontweight='bold', pad=20)
        ax.axis('off')

        # Легенда категорий
        legend_elements = []
        for cat, color in CONFIG.colors.items():
            if cat != 'other':
                legend_elements.append(plt.Line2D([0], [0], marker='o', color='w',
                    markerfacecolor=color, markersize=8, label=cat.capitalize()))

        if legend_elements:
            ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1, 1),
                     fontsize=7, framealpha=0.9)

        plt.tight_layout()
        return fig

# ============================================================================
# 🌐 D3.JS ВИЗУАЛИЗАТОР
# ============================================================================
class D3Visualizer:
    """Визуализация с помощью D3.js"""

    @staticmethod
    def generate_html(graph, node_sizes, layout_type: str = 'force'):
        """Генерация HTML с D3.js"""
        nodes = []
        for node in graph.nodes():
            info = graph.nodes[node].get('info')
            category = graph.nodes[node].get('category', 'other')
            if info and hasattr(info, 'bilingual'):
                bilingual = info.bilingual
                nodes.append({
                    "id": node,
                    "size": node_sizes.get(node, 1),
                    "category": category,
                    "color": CONFIG.colors.get(category, CONFIG.colors['other']),
                    "english": bilingual['english'],
                    "russian": bilingual['russian'],
                    "display": bilingual['display']
                })
            else:
                nodes.append({
                    "id": node, "size": node_sizes.get(node, 1),
                    "category": category, "color": CONFIG.colors.get(category, CONFIG.colors['other']),
                    "english": node, "russian": node
                })

        links = []
        for u, v, d in graph.edges(data=True):
            links.append({"source": u, "target": v, "weight": d.get('weight', 1),
                         "count": d.get('count', 1), "name": d.get('name', '')})

        graph_data = {"nodes": nodes, "links": links}
        category_stats = defaultdict(int)
        for node in nodes:
            category_stats[node['category']] += 1

        return f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>ERD Visualization - Interactive</title>
<script src="https://d3js.org/d3.v7.min.js"></script>
<style>
body {{ margin: 0; font-family: 'Segoe UI', Arial, sans-serif; background: #f5f5f5; }}
#controls {{
    position: fixed; top: 20px; right: 20px; background: white;
    padding: 20px; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.15);
    z-index: 1000; width: 300px; max-height: 90vh; overflow-y: auto;
}}
.control-title {{ margin: 0 0 15px 0; font-size: 16px; font-weight: 600; color: #333; }}
.btn {{
    width: 100%; margin: 6px 0; padding: 10px; border: none; border-radius: 6px;
    cursor: pointer; font-size: 13px; font-weight: 500; transition: all 0.2s;
}}
.btn-primary {{ background: #4CAF50; color: white; }}
.btn-secondary {{ background: #2196F3; color: white; }}
.btn:hover {{ transform: translateY(-2px); box-shadow: 0 4px 12px rgba(0,0,0,0.2); }}
.filter-section {{ margin: 15px 0; }}
.filter-title {{ font-size: 13px; font-weight: 600; margin-bottom: 8px; color: #555; }}
.category-filter {{
    display: flex; flex-wrap: wrap; gap: 5px; margin: 8px 0;
}}
.cat-btn {{
    padding: 5px 10px; border: 1px solid #ddd; border-radius: 15px;
    cursor: pointer; font-size: 11px; transition: all 0.2s;
}}
.cat-btn.active {{ background: #4CAF50; color: white; border-color: #4CAF50; }}
.search-box {{
    width: 100%; padding: 10px; margin: 10px 0; border: 2px solid #e0e0e0;
    border-radius: 6px; font-size: 13px; box-sizing: border-box;
}}
.search-box:focus {{ outline: none; border-color: #4CAF50; }}
#tooltip {{
    position: absolute; background: rgba(0,0,0,0.95); color: white;
    padding: 12px 16px; border-radius: 8px; font-size: 12px; pointer-events: none;
    opacity: 0; transition: opacity 0.2s; max-width: 320px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.3); z-index: 2000;
}}
.tooltip-title {{ font-size: 13px; font-weight: bold; margin-bottom: 8px;
    padding-bottom: 5px; border-bottom: 1px solid #444; }}
.tooltip-row {{ display: flex; margin: 4px 0; font-size: 11px; }}
.tooltip-label {{ width: 80px; color: #aaa; }}
.tooltip-value {{ flex: 1; color: white; word-break: break-word; }}
.stats {{ margin-top: 15px; padding: 12px; background: #f8f9fa;
    border-radius: 8px; font-size: 12px; }}
.stat-row {{ display: flex; justify-content: space-between; margin: 4px 0;
    padding: 4px 0; border-bottom: 1px dashed #ddd; }}
.node {{ stroke: #fff; stroke-width: 2px; cursor: pointer;
    transition: stroke-width 0.2s; }}
.node:hover {{ stroke: #333; stroke-width: 3px; }}
.link {{ stroke: #999; stroke-opacity: 0.5; transition: stroke-opacity 0.2s; }}
.link:hover {{ stroke-opacity: 1; }}
.label {{ font-size: 10px; font-weight: 500; pointer-events: none;
    text-shadow: 0 1px 0 white; }}
.legend {{ margin-top: 10px; }}
.legend-item {{ display: flex; align-items: center; margin: 4px 0; font-size: 11px; }}
.legend-color {{ width: 12px; height: 12px; border-radius: 2px; margin-right: 8px; }}
</style>
</head>
<body>
<div id="controls">
    <div class="control-title">🎮 Управление графом</div>
    <input type="text" class="search-box" id="search" placeholder="🔍 Поиск..." oninput="search(this.value)">
    <div class="filter-section">
        <div class="filter-title">🏷️ Категории:</div>
        <div class="category-filter" id="categoryFilter"></div>
    </div>
    <button class="btn btn-primary" onclick="zoomIn()">➕ Увеличить</button>
    <button class="btn btn-primary" onclick="zoomOut()">➖ Уменьшить</button>
    <button class="btn btn-primary" onclick="fitToScreen()">📏 По размеру</button>
    <button class="btn btn-secondary" onclick="exportSVG()">💾 Экспорт SVG</button>
    <button class="btn btn-secondary" onclick="toggleLabels()">🏷️ Подписи</button>
    <div class="stats">
        <div class="stat-row"><span>📊 Сущностей:</span><span id="stat-nodes">{graph.number_of_nodes()}</span></div>
        <div class="stat-row"><span>🔗 Связей:</span><span id="stat-links">{graph.number_of_edges()}</span></div>
    </div>
    <div class="legend" id="legend"></div>
</div>
<div id="tooltip"></div>
<svg width="100%" height="100vh"></svg>
<script>
const graphData = {json.dumps(graph_data)};
const categoryColors = {json.dumps(dict(CONFIG.colors))};
const categoryStats = {json.dumps(dict(category_stats))};
let showLabels = true;
let activeCategories = new Set(Object.keys(categoryStats));
const width = window.innerWidth;
const height = window.innerHeight;
const svg = d3.select("svg");
const container = svg.append("g");
const zoom = d3.zoom().scaleExtent([0.1, 4]).on("zoom", (event) => {{
    container.attr("transform", event.transform);
}});
svg.call(zoom);
const simulation = d3.forceSimulation(graphData.nodes)
    .force("link", d3.forceLink(graphData.links).id(d => d.id).distance(150))
    .force("charge", d3.forceManyBody().strength(-400))
    .force("center", d3.forceCenter(width / 2, height / 2))
    .force("collision", d3.forceCollide().radius(d => Math.sqrt(d.size) * 12));
const link = container.append("g").selectAll("line")
    .data(graphData.links).enter().append("line")
    .attr("class", "link")
    .attr("stroke-width", d => Math.sqrt(d.weight))
    .attr("stroke", d => d3.interpolateGreys(0.3 + d.weight/10));
const node = container.append("g").selectAll("circle")
    .data(graphData.nodes).enter().append("circle")
    .attr("class", "node")
    .attr("r", d => Math.sqrt(d.size) * 10)
    .attr("fill", d => d.color)
    .call(d3.drag()
        .on("start", (event, d) => {{
            if (!event.active) simulation.alphaTarget(0.3).restart();
            d.fx = d.x; d.fy = d.y;
        }})
        .on("drag", (event, d) => {{ d.fx = event.x; d.fy = event.y; }})
        .on("end", (event, d) => {{
            if (!event.active) simulation.alphaTarget(0);
            d.fx = null; d.fy = null;
        }}))
    .on("mouseover", (event, d) => {{
        const tooltip = d3.select("#tooltip");
        let content = `<div class="tooltip-title">${{d.id}}</div>`;
        if (d.english) content += `<div class="tooltip-row"><span class="tooltip-label">English:</span><span class="tooltip-value">${{d.english}}</span></div>`;
        if (d.russian) content += `<div class="tooltip-row"><span class="tooltip-label">Русский:</span><span class="tooltip-value">${{d.russian}}</span></div>`;
        content += `<div class="tooltip-row"><span class="tooltip-label">Категория:</span><span class="tooltip-value">${{d.category}}</span></div>`;
        content += `<div class="tooltip-row"><span class="tooltip-label">Связей:</span><span class="tooltip-value">${{d.size}}</span></div>`;
        tooltip.style("opacity", 1).style("left", (event.pageX + 20) + "px")
            .style("top", (event.pageY - 20) + "px").html(content);
    }})
    .on("mouseout", () => {{ d3.select("#tooltip").style("opacity", 0); }});
const label = container.append("g").selectAll("text")
    .data(graphData.nodes).enter().append("text")
    .attr("class", "label")
    .attr("dx", d => Math.sqrt(d.size) * 10 + 4)
    .attr("dy", 4)
    .text(d => d.display || d.id);
simulation.on("tick", () => {{
    link.attr("x1", d => d.source.x).attr("y1", d => d.source.y)
        .attr("x2", d => d.target.x).attr("y2", d => d.target.y);
    node.attr("cx", d => d.x).attr("cy", d => d.y);
    label.attr("x", d => d.x).attr("y", d => d.y);
    node.attr("opacity", d => activeCategories.has(d.category) ? 1 : 0.1);
    link.attr("opacity", d => activeCategories.has(d.source.category) &&
        activeCategories.has(d.target.category) ? 0.5 : 0.05);
    label.attr("opacity", d => activeCategories.has(d.category) ? (showLabels ? 1 : 0) : 0.05);
}});
function search(query) {{
    query = query.toLowerCase();
    node.attr("opacity", d => {{
        const matches = activeCategories.has(d.category) && (
            d.id.toLowerCase().includes(query) ||
            (d.english && d.english.toLowerCase().includes(query)) ||
            (d.russian && d.russian.toLowerCase().includes(query)));
        return matches ? 1 : 0.1;
    }});
}}
function zoomIn() {{ svg.transition().call(zoom.scaleBy, 1.3); }}
function zoomOut() {{ svg.transition().call(zoom.scaleBy, 0.7); }}
function fitToScreen() {{
    const bounds = container.node().getBBox();
    const scale = 0.8 / Math.max(bounds.width / width, bounds.height / height);
    const translate = [width/2 - scale*bounds.x - scale*bounds.width/2,
                       height/2 - scale*bounds.y - scale*bounds.height/2];
    svg.transition().duration(750).call(zoom.transform,
        d3.zoomIdentity.translate(translate[0], translate[1]).scale(scale));
}}
function exportSVG() {{
    const svgElement = document.querySelector('svg');
    const serializer = new XMLSerializer();
    let source = serializer.serializeToString(svgElement);
    source = '<?xml version="1.0" encoding="UTF-8"?>' + source;
    const blob = new Blob([source], {{type: 'image/svg+xml'}});
    const url = URL.createObjectURL(blob);
    const link = document.createElement('a');
    link.href = url; link.download = 'erd_diagram.svg'; link.click();
    URL.revokeObjectURL(url);
}}
function toggleLabels() {{ showLabels = !showLabels; simulation.alpha(0.3).restart(); }}
function initCategoryFilter() {{
    const filterDiv = document.getElementById('categoryFilter');
    const legendDiv = document.getElementById('legend');
    for (const [cat, count] of Object.entries(categoryStats)) {{
        const btn = document.createElement('div');
        btn.className = 'cat-btn active';
        btn.textContent = `${{cat}} (${{count}})`;
        btn.style.borderColor = categoryColors[cat] || '#ccc';
        btn.onclick = () => toggleCategory(cat, btn);
        filterDiv.appendChild(btn);
        const legendItem = document.createElement('div');
        legendItem.className = 'legend-item';
        legendItem.innerHTML = `<div class="legend-color" style="background:${{categoryColors[cat] || '#ccc'}}"></div>${{cat}} (${{count}})`;
        legendDiv.appendChild(legendItem);
    }}
}}
function toggleCategory(cat, btn) {{
    if (activeCategories.has(cat)) {{
        activeCategories.delete(cat);
        btn.classList.remove('active');
    }} else {{
        activeCategories.add(cat);
        btn.classList.add('active');
    }}
    simulation.alpha(0.3).restart();
}}
initCategoryFilter();
</script>
</body>
</html>
"""

# ============================================================================
# 🎯 ОСНОВНОЙ КЛАСС ВИЗУАЛИЗАТОРА
# ============================================================================
class ERDVisualizer:
    """Основной класс для визуализации ERD"""

    def __init__(self):
        self.translator = BilingualTranslator()
        self.parser = ERDParser(self.translator)
        self.graph = None
        self.node_sizes = None
        self.layout_type = 'spring'

    def load(self, content: str, max_nodes: int = 60) -> bool:
        """Загрузка ERD из файла"""
        if self.parser.parse(content):
            builder = GraphBuilder(self.parser, max_nodes)
            self.graph = builder.graph
            self.node_sizes = builder.node_sizes
            return True
        return False

    def set_layout(self, layout: str):
        """Установка типа layout"""
        self.layout_type = layout

    def show_matplotlib(self, layout: str = None):
        """Показать Matplotlib визуализацию"""
        if self.graph is None:
            print("❌ Сначала загрузите ERD файл")
            return
        layout = layout or self.layout_type
        fig = MatplotlibVisualizer.visualize(self.graph, self.node_sizes,
                                            layout, CONFIG.figsize_medium)
        plt.show()
        return fig

    def show_d3(self):
        """Показать D3.js визуализацию"""
        if self.graph is None:
            print("❌ Сначала загрузите ERD файл")
            return
        html = D3Visualizer.generate_html(self.graph, self.node_sizes, self.layout_type)
        display(HTML(html))

    def print_stats(self):
        """Вывод статистики"""
        if self.graph is None:
            print("❌ Сначала загрузите ERD файл")
            return

        stats = GraphBuilder(self.parser, CONFIG.max_nodes_display).get_stats()
        category_stats = TableCategorizer.get_category_stats(self.parser.entities)

        print("\n" + "=" * 80)
        print("📊 СТАТИСТИКА ERD")
        print("=" * 80)
        print(f"📌 Всего таблиц в файле: {len(self.parser.entities)}")
        print(f"📌 Отображено сущностей: {stats['nodes']}")
        print(f"🔗 Всего связей: {stats['edges']}")

        print("\n📁 ТАБЛИЦЫ ПО КАТЕГОРИЯМ:")
        print("-" * 80)
        category_names = {
            'directory': '📚 Справочники', 'log': '📝 Журналы',
            'patient': '👥 Пациенты', 'medical': '🏥 Медицинские',
            'drug': '💊 Лекарства', 'lab': '🔬 Лаборатория',
            'finance': '💰 Финансы', 'admin': '🏢 Админ',
            'report': '📊 Отчёты', 'service': '🛎️ Услуги',
            'oms': '🏥 ОМС', 'parus': '🔄 Парус', 'cp': '📋 Копии',
            'other': '📦 Прочие'
        }
        for cat, count in sorted(category_stats.items(), key=lambda x: x[1], reverse=True):
            name = category_names.get(cat, cat.capitalize())
            print(f"   {name}: {count} таблиц")

        print("\n🔥 ТОП-10 СВЯЗАННЫХ СУЩНОСТЕЙ:")
        print("-" * 80)
        for i, (entity, degree) in enumerate(stats['top_entities'], 1):
            info = self.graph.nodes[entity].get('info')
            if info:
                print(f"{i:2}. {info.display_name}: {degree} связей")
            else:
                print(f"{i:2}. {entity}: {degree} связей")

# ============================================================================
# ИНТЕРФЕЙС ПОЛЬЗОВАТЕЛЯ
# ============================================================================
viz = ERDVisualizer()

upload = widgets.FileUpload(
    accept='.txt,.xml',
    multiple=False,
    description='📁 Загрузить ERD',
    layout=widgets.Layout(width='450px')
)

# 🔴 ИСПРАВЛЕННЫЙ DROPDOWN
layout_select = widgets.Dropdown(
    options=[
        ('Spring (физическая симуляция)', 'spring'),
        ('Circular (круговой)', 'circular'),
        ('Kamada-Kawai (энергетический)', 'kamada_kawai'),
        ('Shell (концентрические круги)', 'shell'),
        ('Spectral (спектральный)', 'spectral'),
        ('Planar (плоскостной)', 'planar'),
        ('Random (случайный)', 'random'),
        ('Bipartite (двудольный)', 'bipartite')
    ],
    value='spring',
    description='📐 Layout:',
    layout=widgets.Layout(width='450px')
)

max_nodes = widgets.IntSlider(
    value=50,
    min=20,
    max=100,
    step=10,
    description='Макс. сущностей:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px')
)

viz_type = widgets.RadioButtons(
    options=['📊 Matplotlib', '🌐 Интерактивный D3', '🔄 Оба'],
    value='🔄 Оба',
    description='Тип:',
    layout=widgets.Layout(width='450px')
)

load_btn = widgets.Button(
    description='📊 Загрузить и визуализировать',
    button_style='primary',
    layout=widgets.Layout(width='220px')
)
export_btn = widgets.Button(
    description='📝 Экспорт переводов',
    button_style='info',
    layout=widgets.Layout(width='220px')
)
png_btn = widgets.Button(
    description='💾 Экспорт PNG',
    button_style='success',
    layout=widgets.Layout(width='220px')
)
stats_btn = widgets.Button(
    description='📈 Статистика',
    button_style='warning',
    layout=widgets.Layout(width='220px')
)

output = widgets.Output()
stats_output = widgets.Output()

# Обработчики событий
def on_load(b):
    with output:
        clear_output()
        if not upload.value:
            print("⚠️ Пожалуйста, загрузите файл")
            return

        start_time = time.time()
        file = list(upload.value.values())[0]
        content = file['content'].decode('utf-8')

        print("🔄 Парсинг ERD файла...")
        viz.set_layout(layout_select.value)  # 🔴 Теперь value корректный
        if viz.load(content, max_nodes.value):
            elapsed = time.time() - start_time
            print(f"✅ Загружено за {elapsed:.2f}с")
            print(f"📌 Сущностей: {viz.graph.number_of_nodes()}")
            print(f"🔗 Связей: {viz.graph.number_of_edges()}")
            print(f"📐 Layout: {layout_select.value}")

            with stats_output:
                clear_output()
                viz.print_stats()

            if viz_type.value in ['📊 Matplotlib', '🔄 Оба']:
                print(f"\n📊 Генерация Matplotlib ({layout_select.value})...")
                viz.show_matplotlib()

            if viz_type.value in ['🌐 Интерактивный D3', '🔄 Оба']:
                print("\n🌐 Генерация интерактивной D3 визуализации...")
                viz.show_d3()
        else:
            print("❌ Ошибка загрузки файла")

def on_export(b):
    with output:
        clear_output()
        data = []
        for entity in viz.parser.entities.values():
            data.append({
                'original': entity.name,
                'english': entity.bilingual['english'],
                'russian': entity.bilingual['russian'],
                'display': entity.bilingual['display'],
                'category': entity.category
            })
        df = pd.DataFrame(data)
        df.to_csv("translations.csv", index=False, encoding='utf-8-sig')
        print(f"✅ Экспортировано {len(df)} переводов в translations.csv")
        files.download("translations.csv")

def on_png(b):
    with output:
        clear_output()
        if viz.graph:
            fig = MatplotlibVisualizer.visualize(
                viz.graph, viz.node_sizes,
                viz.layout_type, CONFIG.figsize_large
            )
            fig.savefig("erd_diagram.png", dpi=300, bbox_inches='tight', facecolor='white')
            plt.close(fig)
            print("✅ PNG сохранен: erd_diagram.png")
            files.download("erd_diagram.png")

def on_stats(b):
    with stats_output:
        clear_output()
        viz.print_stats()

# Привязка обработчиков
load_btn.on_click(on_load)
export_btn.on_click(on_export)
png_btn.on_click(on_png)
stats_btn.on_click(on_stats)

# Отображение интерфейса
print("=" * 80)
print("🎯 УЛУЧШЕННЫЙ ERD ВИЗУАЛИЗАТОР С ГРУППИРОВКОЙ ПО КАТЕГОРИЯМ")
print("=" * 80)
print("📁 Формат: English | Русский")
print("🎨 Категории: Справочники, Журналы, Пациенты, Медицинские...")
print("📐 Layout: Spring, Circular, Kamada-Kawai, Shell...")
print("=" * 80)

display(widgets.VBox([
    upload,
    layout_select,
    max_nodes,
    viz_type,
    widgets.HBox([load_btn, export_btn, png_btn, stats_btn]),
    stats_output,
    output
]))

print("\n📌 ИНСТРУКЦИЯ:")
print("1. Загрузите файл DB_MIS_REPORT.txt")
print("2. Выберите layout для графа")
print("3. Настройте количество отображаемых сущностей")
print("4. Нажмите 'Загрузить и визуализировать'")
print("\n🎮 ИНТЕРАКТИВНЫЙ РЕЖИМ:")
print("   • Фильтруйте по категориям таблиц")
print("   • Поиск по названиям (рус/eng)")
print("   • Перетаскивание узлов, зум")
print("   • Экспорт в SVG")

🎯 УЛУЧШЕННЫЙ ERD ВИЗУАЛИЗАТОР С ГРУППИРОВКОЙ ПО КАТЕГОРИЯМ
📁 Формат: English | Русский
🎨 Категории: Справочники, Журналы, Пациенты, Медицинские...
📐 Layout: Spring, Circular, Kamada-Kawai, Shell...



📌 ИНСТРУКЦИЯ:
1. Загрузите файл DB_MIS_REPORT.txt
2. Выберите layout для графа
3. Настройте количество отображаемых сущностей
4. Нажмите 'Загрузить и визуализировать'

🎮 ИНТЕРАКТИВНЫЙ РЕЖИМ:
   • Фильтруйте по категориям таблиц
   • Поиск по названиям (рус/eng)
   • Перетаскивание узлов, зум
   • Экспорт в SVG
